# BanglaSlumNet × LocateAnything — Master Colab Notebook
**RESEARCH-ONLY.** See THIRD_PARTY_LICENSES.md.

## How to read each cell header
Every code cell starts with a comment block:
```
# [GPU]  recommended runtime (CPU is fine / A100 / any GPU)
# [TIME] estimated wall-clock
# [CU]   compute-unit cost (one-time vs per-run; ~0 = negligible)
```
**CU-saving design:** every heavy phase writes to Google Drive and checks a cache/
sentinel/registry first, so re-running skips finished work. A disconnect costs you
nothing already-computed. To minimise CU: **run Phase 0 on a CPU runtime**, switch to
**A100** only for Phases 1–4, then switch back to CPU for Phase 5.

## Total one-time CU budget (A100 40GB, per spec §9.7)
| Phase | One-time? | A100 time |
|---|---|---|
| P1 model download + introspection | yes | 15–35 min |
| P2 LocateAnything label validation | yes | 30–50 min |
| P3 MoonViT feature extraction | yes | 30–60 min |
| P3 SAS-Net training | yes | ≤ 1 hr |
| P4 all experiments (cached features) | per-run | 1–2 hr total |
| **Total to full results** | | **~5–7 A100-hr** |

**Run `00_smoke_test.ipynb` first (CPU, ~5 min, 0 CU) before anything here.**

## Phase 0 — Setup  *(run on a CPU runtime to save CU)*

In [ ]:
# [GPU]  CPU is fine
# [TIME] ~1 min
# [CU]   ~0 (no GPU)
# Cell P0.1: Mount Drive and set PROJECT_ROOT (all caches/results live here)
import sys, os, subprocess
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/gdrive')
    PROJECT_ROOT = Path('/gdrive/MyDrive/BanglaSlumNet')
else:
    PROJECT_ROOT = Path('.').resolve().parent

PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
print(f'PROJECT_ROOT (results persist here): {PROJECT_ROOT}')

In [ ]:
# [GPU]  CPU is fine
# [TIME] ~1 min
# [CU]   ~0
# Cell P0.2: Clone or pull repo
REPO_DIR = PROJECT_ROOT / 'repo'
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', 'https://github.com/namaray/BanglaSlumNet.git', str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull'], check=True)
os.chdir(str(REPO_DIR))
sys.path.insert(0, str(REPO_DIR))
print(f'Repo at: {REPO_DIR}')

In [ ]:
# [GPU]  CPU is fine
# [TIME] ~3–5 min (first time only; pip caches afterwards)
# [CU]   ~0 on CPU runtime
# Cell P0.3: Install requirements (Colab order). DO NOT install MagiAttention (Ampere unsupported).
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements_colab.txt'], check=True)
print('Requirements installed.')

In [ ]:
# [GPU]  any (switch to A100 now if continuing to Phase 1+)
# [TIME] instant
# [CU]   ~0
# Cell P0.4: GPU check (warn if < 24 GB)
import torch
subprocess.run(['nvidia-smi'], check=False)
if torch.cuda.is_available():
    gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {torch.cuda.get_device_name(0)} | {gb:.1f} GB')
    if gb < 24:
        print('WARNING: < 24 GB VRAM. Enable cfg.locate_anything.load_in_4bit = True below.')
    DEVICE = 'cuda'
else:
    print('No GPU. Phases 1–4 need a GPU runtime; Phases 0 and 5 are fine on CPU.')
    DEVICE = 'cpu'

In [ ]:
# [GPU]  CPU is fine
# [TIME] instant
# [CU]   ~0
# Cell P0.5: Load config + Decision flags (§15). Edit flags HERE, never in source.
from omegaconf import OmegaConf
cfg = OmegaConf.load('config/default.yaml')

cfg.decide = OmegaConf.create({
    'D1_feature_mode': 'hidden_state',   # hidden_state | grounding_map (auto-fallback)
    'D2_lora_adapt': False,
    'D3_atm_correction': 'classical',
    'D4_generalization': 'loro',
    'D6_include_gram': True,
    'D7_use_swir': False,
})

# Route ALL caches and results to Drive so they survive disconnects and you can pull them locally.
cfg.paths.project_root = str(PROJECT_ROOT)
cfg.paths.data_dir = str(PROJECT_ROOT / 'data')
cfg.paths.tiles_dir = str(PROJECT_ROOT / 'data/tiles')
cfg.paths.labels_dir = str(PROJECT_ROOT / 'data/labels')
cfg.paths.socioeconomic_dir = str(PROJECT_ROOT / 'data/socioeconomic')
cfg.paths.features_cache_dir = str(PROJECT_ROOT / 'data/features_cache')
cfg.paths.results_dir = str(PROJECT_ROOT / 'results')
cfg.paths.manifest = str(PROJECT_ROOT / 'data/dataset_manifest.json')
cfg.paths.confidence_json = str(PROJECT_ROOT / 'data/labels/confidence.json')
cfg.paths.la_validation_json = str(PROJECT_ROOT / 'data/labels/la_validation.json')
cfg.paths.registry = str(PROJECT_ROOT / 'results/runs/registry.json')
cfg.locate_anything.feature_mode = cfg.decide.D1_feature_mode
for p in [cfg.paths.data_dir, cfg.paths.tiles_dir, cfg.paths.labels_dir,
          cfg.paths.socioeconomic_dir, cfg.paths.features_cache_dir,
          str(PROJECT_ROOT/'results/runs'), str(PROJECT_ROOT/'results/figures'),
          str(PROJECT_ROOT/'results/tables')]:
    Path(p).mkdir(parents=True, exist_ok=True)
print(OmegaConf.to_yaml(cfg))

## Phase 1 — Models & Data  *(switch to A100 now)*

In [ ]:
# [GPU]  A100 / L4 recommended (network-bound, but you want the GPU session ready)
# [TIME] 15–30 min FIRST TIME ONLY; instant once cached to Drive
# [CU]   one-time. Skips entirely on re-run via sentinel flag.
# Cell P1.1: Download LocateAnything-3B to Drive (idempotent)
MODEL_CACHE = str(PROJECT_ROOT / 'model_cache')
SENTINEL = PROJECT_ROOT / 'model_cache' / 'download_done.flag'
if not SENTINEL.exists():
    subprocess.run([sys.executable, 'scripts/download_models.py', '--cache_dir', MODEL_CACHE], check=True)
    SENTINEL.touch()
else:
    print('Model cache exists — skipping download (0 CU).')
cfg.locate_anything['cache_dir'] = MODEL_CACHE

In [ ]:
# [GPU]  A100 (loads the 4B model in BF16, ~10 GB)
# [TIME] ~3–5 min (model load dominates)
# [CU]   one-time; run ONCE to confirm the feature-extractor encoder path.
# Cell P1.2: Inspect model internals — CRITICAL for feature extraction wiring.
# If the printed vision-encoder attribute path is NOT one of _VISION_ENCODER_CANDIDATES
# in src/locate_anything/worker.py, add it there, or set D1_feature_mode='grounding_map'.
from src.locate_anything.worker import LocateAnythingWorker
worker = LocateAnythingWorker(
    cache_dir=MODEL_CACHE, device=DEVICE,
    load_in_4bit=cfg.locate_anything.load_in_4bit,
)
worker.load()
worker.inspect_model_internals()
print('\nTODO_VERIFY: confirm vision-encoder path above matches worker.py assumptions.')

In [ ]:
# [GPU]  CPU is fine
# [TIME] ~2 min
# [CU]   ~0
# Cell P1.3: Region preview on map — VERIFY all TODO_VERIFY bounding boxes here.
import yaml
with open('config/regions_dhaka.yaml') as f:
    regions_cfg = yaml.safe_load(f)
for name, r in regions_cfg['regions'].items():
    bb = r['bbox']
    print(f'  {name}: [{bb["lon_min"]}, {bb["lat_min"]}, {bb["lon_max"]}, {bb["lat_max"]}]')
try:
    import folium
    m = folium.Map(location=[23.78, 90.41], zoom_start=12)
    colors = ['red', 'blue', 'green', 'orange', 'purple']
    for (name, r), color in zip(regions_cfg['regions'].items(), colors):
        bb = r['bbox']
        folium.Rectangle(bounds=[[bb['lat_min'], bb['lon_min']], [bb['lat_max'], bb['lon_max']]],
                         color=color, fill=True, fill_opacity=0.2, tooltip=name).add_to(m)
    display(m)
except ImportError:
    print('pip install folium for the visual map preview.')

In [ ]:
# [GPU]  CPU is fine (GEE does the compute server-side; this only downloads results)
# [TIME] 10–40 min the first time (depends on #regions × years × seasons); skips existing
# [CU]   ~0 on Colab (no local GPU/compute). GEE quota is separate from Colab CU.
# Cell P1.4: GEE EXPORTS via the Python `ee` API — runs IN THIS NOTEBOOK.
# This is the in-notebook alternative to pasting gee/*.js into the GEE Code Editor.
# Files download straight into the Drive-mounted data dirs and are cache-skipped.
GEE_PROJECT = 'banglaslumnet-research'   # <-- your Earth Engine project id
RUN_GEE_EXPORTS = True                    # set False to skip (e.g. already exported)

if not RUN_GEE_EXPORTS:
    print('GEE export skipped (RUN_GEE_EXPORTS=False).')
else:
    from gee.ee_export_utils import init_ee
    from gee.export_s2_composites import run_all as export_s2
    from gee.export_weak_labels import run_all as export_labels
    from gee.export_socioeconomic import run_all as export_socioec

    init_ee(GEE_PROJECT)   # triggers ee.Authenticate() on first run in a fresh runtime

    REGIONS_YAML = 'config/regions_dhaka.yaml'
    ref_year = cfg.data.training_years[0]

    print('1/3 Sentinel-2 composites ...')
    export_s2(output_dir=cfg.paths.tiles_dir, regions_yaml=REGIONS_YAML,
              years=list(cfg.data.training_years), seasons=list(cfg.data.seasons),
              bands=list(cfg.data.s2_bands))

    print('2/3 Weak labels ...')
    export_labels(output_dir=cfg.paths.labels_dir, regions_yaml=REGIONS_YAML,
                  reference_year=ref_year,
                  viirs_percentile=cfg.weak_labels.viirs_percentile_threshold)

    print('3/3 Socioeconomic E-tensor ...')
    export_socioec(output_dir=cfg.paths.socioeconomic_dir, regions_yaml=REGIONS_YAML,
                   channels=list(cfg.fusion.socioeconomic_channels), reference_year=ref_year)

    print('\nAll GEE exports landed in:', cfg.paths.data_dir)

# Optional: ESRI z16 tiles for the GRAM baseline (separate, not needed for BanglaSlumNet)
# !python gee/02_export_esri_tiles.py --output_dir {cfg.paths.tiles_dir}

In [ ]:
# [GPU]  CPU is fine
# [TIME] ~2–5 min (depends on tile count)
# [CU]   ~0 (CPU raster ops). Idempotent: already-tiled tiles are skipped.
# Cell P1.5: Tile regional GeoTIFFs -> aligned 256x256 per-tile stacks + manifest.
# Pass 1 (no LocateAnything yet): produces rgb/noisy/hcgeo/hc/socioec tiles.
# Reprojects labels+socioec onto the RGB grid, so misalignment is impossible.
import json
from src.data.tiling import tile_all_regions
cfg_dict = OmegaConf.to_container(cfg, resolve=True)
if Path(cfg.paths.manifest).exists():
    with open(cfg.paths.manifest) as f:
        manifest = json.load(f)
    print(f'Manifest exists: {manifest["n_tiles"]} tiles. Re-run tiling only adds new tiles.')
tile_all_regions(cfg_dict, regions_yaml='config/regions_dhaka.yaml',
                 reference_year=cfg.data.training_years[0],
                 reference_season=cfg.data.seasons[0], la_validation=None)
with open(cfg.paths.manifest) as f:
    print('Total tiles:', json.load(f)['n_tiles'])

## Phase 2 — Weak Labels  *(A100; one-time, cached)*

In [ ]:
# [GPU]  CPU is fine
# [TIME] ~2–5 min
# [CU]   ~0 (CPU raster ops)
# Cell P2.1: Ingest GEE weak-label rasters → confidence.json (skips if present)
from src.data.weak_labels import build_confidence_json
import glob
if Path(cfg.paths.confidence_json).exists():
    print('confidence.json exists — skipping geospatial fusion.')
else:
    label_tifs = glob.glob(str(Path(cfg.paths.labels_dir) / 'weeklabels_*.tif'))
    if not label_tifs:
        print('No label TIFs found. Finish GEE export first (P1.4).')
    else:
        build_confidence_json(label_tifs=label_tifs, output_labels_dir=cfg.paths.labels_dir,
                              la_validation_path=cfg.paths.la_validation_json, use_la=False,
                              output_confidence_path=cfg.paths.confidence_json)

In [ ]:
# [GPU]  A100 strongly recommended (slow-mode VLM grounding)
# [TIME] 30–50 min for ~1000 tiles (one-time); tqdm bar, saves every 10 tiles
# [CU]   ONE-TIME. Resume-safe: re-run continues from where it stopped.
# Cell P2.2: LocateAnything zero-shot label validation (Direction B). Cached to Drive.
if Path(cfg.paths.la_validation_json).exists():
    import json
    with open(cfg.paths.la_validation_json) as f:
        done = len(json.load(f).get('tiles', []))
    print(f'la_validation.json exists ({done} tiles). Re-run only resumes missing tiles.')
if not cfg.weak_labels.use_locate_anything_validation:
    print('LA validation disabled (use_locate_anything_validation=false). HC stays 3-signal.')
else:
    from src.locate_anything.label_validator import run_label_validation, load_validation
    from src.data.tiling import refresh_hc_with_la
    import json, rasterio, numpy as np
    from PIL import Image
    with open(cfg.paths.manifest) as f:
        tile_ids = [t['tile_id'] for t in json.load(f)['tiles']]

    def esri_image_loader(tile_id):
        p = Path(cfg.paths.tiles_dir) / f'{tile_id}_esri.png'
        if p.exists():
            return Image.open(str(p)).convert('RGB')
        p2 = Path(cfg.paths.tiles_dir) / f'{tile_id}_rgb.tif'
        with rasterio.open(str(p2)) as src:
            arr = src.read([1, 2, 3]).transpose(1, 2, 0)
        return Image.fromarray((arr * 255).clip(0, 255).astype('uint8'))

    run_label_validation(worker=worker, tile_ids=tile_ids, image_loader=esri_image_loader,
                         output_path=cfg.paths.la_validation_json,
                         generation_mode=cfg.locate_anything.generation_mode,
                         max_new_tokens=cfg.locate_anything.max_new_tokens, resume=True)

    # Promote per-tile HC masks to the 4-signal version (reads preserved hcgeo; idempotent).
    la = load_validation(cfg.paths.la_validation_json)
    cfg_dict = OmegaConf.to_container(cfg, resolve=True)
    refresh_hc_with_la(cfg_dict, la, manifest_path=cfg.paths.manifest)

    # Rebuild confidence.json with the 4th signal for the data card / figure.
    import glob
    from src.data.weak_labels import build_confidence_json
    label_tifs = glob.glob(str(Path(cfg.paths.labels_dir) / 'weeklabels_*.tif'))
    if label_tifs:
        build_confidence_json(label_tifs=label_tifs, output_labels_dir=cfg.paths.labels_dir,
                              la_validation_path=cfg.paths.la_validation_json, use_la=True,
                              output_confidence_path=cfg.paths.confidence_json)

In [ ]:
# [GPU]  CPU is fine
# [TIME] ~1 min
# [CU]   ~0
# Cell P2.3: Render confidence-strata figure (saved to Drive results/figures)
from src.viz.plots import fig_confidence_strata
import json
if Path(cfg.paths.confidence_json).exists():
    fig_confidence_strata(confidence_json=cfg.paths.confidence_json,
                          output_dir=str(Path(cfg.paths.results_dir) / 'figures'))
    with open(cfg.paths.confidence_json) as f:
        conf = json.load(f)
    print(f'Total HC pixels: {sum(t.get("n_hc", 0) for t in conf.get("tiles", [])):,}')
else:
    print('Run P2.1 first.')

## Phase 3 — Feature & SAS-Net Caching  *(A100; one-time)*

In [ ]:
# [GPU]  A100 required (frozen MoonViT forward passes)
# [TIME] 30–60 min one-time; tqdm per-tile bar; skips cached tiles
# [CU]   ONE-TIME, the single most expensive repeated op — cached as .npy, never re-run.
# Cell P3.1: MoonViT feature extraction. Extracts BOTH prompt sets so every config is covered:
#   'full'       -> slum + formal prompt features (vlm_lang, full)
#   'vlm_visual' -> neutral prompt features (vlm_visual, exp1)
FEAT_DONE = Path(cfg.paths.features_cache_dir) / 'extraction_done.flag'
if FEAT_DONE.exists():
    print('Feature cache exists — skipping (0 CU).')
else:
    from src.locate_anything.feature_extractor import FeatureExtractor
    import json, rasterio, numpy as np
    from PIL import Image
    extractor = FeatureExtractor(worker=worker, cache_dir=cfg.paths.features_cache_dir,
                                 feature_mode=cfg.locate_anything.feature_mode,
                                 generation_mode=cfg.locate_anything.generation_mode,
                                 max_new_tokens=cfg.locate_anything.max_new_tokens)
    with open(cfg.paths.manifest) as f:
        all_tile_ids = [t['tile_id'] for t in json.load(f)['tiles']]
    def rgb_loader(tile_id):
        p = Path(cfg.paths.tiles_dir) / f'{tile_id}_rgb.tif'
        with rasterio.open(str(p)) as src:
            arr = src.read([1, 2, 3]).transpose(1, 2, 0)
        return Image.fromarray(np.clip(arr * 255, 0, 255).astype('uint8'))
    for mc in ['full', 'vlm_visual']:
        print(f'Extracting features for config: {mc}')
        extractor.extract_all_tiles(all_tile_ids, rgb_loader, model_config=mc, verbose=True)
    FEAT_DONE.touch()

In [ ]:
# [GPU]  A100 (or L4)
# [TIME] ≤ 1 hr one-time; tqdm per-epoch loss bar; caches clean tiles
# [CU]   ONE-TIME. Skips if checkpoint + clean-tile flag exist.
# Cell P3.2: Train SAS-Net (Stage 1) on RAW tiles, then cache clean (reference) tiles.
SASNET_CKPT = Path(cfg.paths.runs_dir) / 'sasnet_best.pt'
CLEAN_FLAG = Path(cfg.paths.tiles_dir) / 'clean_tiles_done.flag'
if SASNET_CKPT.exists() and CLEAN_FLAG.exists():
    print('SAS-Net + clean tiles cached — skipping (0 CU).')
else:
    from src.data.tiles import SlumTileDataset
    from src.data.augment import get_train_transform
    from torch.utils.data import DataLoader
    from src.train.train_sasnet import train_sasnet, cache_clean_tiles
    cfg_dict = OmegaConf.to_container(cfg, resolve=True)
    eco = cfg_dict['fusion']['socioeconomic_channels']
    # use_clean_tiles=False: SAS-Net must see raw imagery, not its own output.
    tr = SlumTileDataset(cfg.paths.manifest, 'train', cfg.paths.tiles_dir, cfg.paths.labels_dir,
                         cfg.paths.socioeconomic_dir, eco, transform=get_train_transform(),
                         use_clean_tiles=False)
    va = SlumTileDataset(cfg.paths.manifest, 'val', cfg.paths.tiles_dir, cfg.paths.labels_dir,
                         cfg.paths.socioeconomic_dir, eco, use_clean_tiles=False)
    bs = cfg_dict['train']['batch_size']
    ckpt = train_sasnet(cfg_dict, DataLoader(tr, batch_size=bs, shuffle=True, num_workers=2),
                        DataLoader(va, batch_size=bs, num_workers=2),
                        output_dir=cfg.paths.runs_dir, device=DEVICE)
    # Cache clean tiles for ALL splits so every downstream tile has a clean version.
    allds = SlumTileDataset(cfg.paths.manifest, 'train', cfg.paths.tiles_dir, cfg.paths.labels_dir,
                            cfg.paths.socioeconomic_dir, eco, use_clean_tiles=False)
    for split in ['val', 'test']:
        sp = SlumTileDataset(cfg.paths.manifest, split, cfg.paths.tiles_dir, cfg.paths.labels_dir,
                             cfg.paths.socioeconomic_dir, eco, use_clean_tiles=False)
        allds.tiles += sp.tiles
    cache_clean_tiles(ckpt, DataLoader(allds, batch_size=8, num_workers=2),
                      cfg.paths.tiles_dir, cfg_dict, DEVICE)
    CLEAN_FLAG.touch()

## Phase 4 — Experiments  *(A100; cheap & resumable over cached features)*

In [ ]:
# [GPU]  CPU is fine for this cell
# [TIME] instant
# [CU]   ~0
# Cell P4.1: Load experiment matrix + registry
import yaml
from src.tracking.registry import RunRegistry
with open('config/experiments.yaml') as f:
    experiments = yaml.safe_load(f)['experiments']
registry = RunRegistry(cfg.paths.registry)
for exp in experiments:
    registry.register(exp['id'], 'tbd', exp['experiment'])
print(f'{len(experiments)} experiment rows | registry: {registry.summary()}')
FORCE_RERUN = False  # True only to recompute already-DONE rows

In [ ]:
# [GPU]  A100 (or L4); trains tiny heads over CACHED features — no VLM calls here
# [TIME] 5–20 min per row; 1–2 hr for the full matrix. tqdm per-epoch bar + outer tqdm.
# [CU]   per-run, but cheap. DONE rows skip instantly (registry-gated) — disconnect-safe.
# Cell P4.2: Orchestration loop — results JSON + checkpoints written to Drive per row.
from src.train.train_segmenter import train_segmenter
from src.eval.evaluate import evaluate
from src.data.tiles import SlumTileDataset
from src.data.augment import get_train_transform
from src.data.preflight import detect_feature_dim
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
import copy
base_cfg = OmegaConf.to_container(cfg, resolve=True)

for exp in tqdm(experiments, desc='Experiment matrix', unit='run'):
    run_id = exp['id']
    if not registry.should_run(run_id, force=FORCE_RERUN):
        print(f'[{run_id}] done — skip')
        continue
    print(f'\n[{run_id}] {exp["description"]}')
    registry.set_status(run_id, 'running')
    try:
        run_cfg = copy.deepcopy(base_cfg)
        for key, val in exp.get('overrides', {}).items():
            d = run_cfg
            ks = key.split('.')
            for k in ks[:-1]:
                d = d.setdefault(k, {})
            d[ks[-1]] = val
        run_cfg['_experiment_id'] = exp['experiment']
        mc = run_cfg.get('model', {}).get('config', 'full')
        eco = run_cfg.get('fusion', {}).get('socioeconomic_channels', [])

        # Set feature_dim from the actual cached features for this config (no guessing).
        if mc != 'baseline_cnn':
            run_cfg.setdefault('locate_anything', {})['feature_dim'] = \
                detect_feature_dim(cfg.paths.features_cache_dir, mc)

        common = dict(tiles_dir=cfg.paths.tiles_dir, labels_dir=cfg.paths.labels_dir,
                      socioeconomic_dir=cfg.paths.socioeconomic_dir, socioeconomic_channels=eco,
                      features_cache_dir=cfg.paths.features_cache_dir, model_config=mc)
        tr = SlumTileDataset(cfg.paths.manifest, 'train', transform=get_train_transform(), **common)
        va = SlumTileDataset(cfg.paths.manifest, 'val', **common)
        te = SlumTileDataset(cfg.paths.manifest, 'test', use_hc_only=True, **common)
        bs = run_cfg.get('train', {}).get('batch_size', 16)
        ckpt = train_segmenter(run_cfg, run_id,
                               DataLoader(tr, batch_size=bs, shuffle=True, num_workers=2),
                               DataLoader(va, batch_size=bs, num_workers=2),
                               output_dir=str(Path(cfg.paths.runs_dir)/run_id), device=DEVICE)
        m = evaluate(run_cfg, run_id, ckpt, DataLoader(te, batch_size=bs, num_workers=2),
                     results_dir=cfg.paths.results_dir, device=DEVICE)
        registry.set_status(run_id, 'done', result_path=str(Path(cfg.paths.runs_dir)/run_id))
        print(f'[{run_id}] HC-IoU={m.get("hc_iou", float("nan")):.4f}')
    except Exception as e:
        registry.set_status(run_id, 'failed')
        import traceback; traceback.print_exc()
print(f'\nDone. Registry: {registry.summary()}')

In [ ]:
# [GPU]  A100 (or L4); trains tiny heads over CACHED features — no VLM calls here
# [TIME] 5–20 min per row; 1–2 hr for the full matrix. tqdm per-epoch bar + outer tqdm.
# [CU]   per-run, but cheap. DONE rows skip instantly (registry-gated) — disconnect-safe.
# Cell P4.2: Orchestration loop — results JSON + checkpoints written to Drive per row.
from src.train.train_segmenter import train_segmenter
from src.eval.evaluate import evaluate
from src.data.tiles import SlumTileDataset
from src.data.augment import get_train_transform
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
import copy
base_cfg = OmegaConf.to_container(cfg, resolve=True)

for exp in tqdm(experiments, desc='Experiment matrix', unit='run'):
    run_id = exp['id']
    if not registry.should_run(run_id, force=FORCE_RERUN):
        print(f'[{run_id}] done — skip')
        continue
    print(f'\n[{run_id}] {exp["description"]}')
    registry.set_status(run_id, 'running')
    try:
        run_cfg = copy.deepcopy(base_cfg)
        for key, val in exp.get('overrides', {}).items():
            d = run_cfg
            ks = key.split('.')
            for k in ks[:-1]:
                d = d.setdefault(k, {})
            d[ks[-1]] = val
        run_cfg['_experiment_id'] = exp['experiment']
        eco = run_cfg.get('fusion', {}).get('socioeconomic_channels', [])
        common = dict(tiles_dir=cfg.paths.tiles_dir, labels_dir=cfg.paths.labels_dir,
                      socioeconomic_dir=cfg.paths.socioeconomic_dir, socioeconomic_channels=eco)
        tr = SlumTileDataset(cfg.paths.manifest, 'train', transform=get_train_transform(), **common)
        va = SlumTileDataset(cfg.paths.manifest, 'val', **common)
        te = SlumTileDataset(cfg.paths.manifest, 'test', use_hc_only=True, **common)
        bs = run_cfg.get('train', {}).get('batch_size', 16)
        ckpt = train_segmenter(run_cfg, run_id,
                               DataLoader(tr, batch_size=bs, shuffle=True, num_workers=2),
                               DataLoader(va, batch_size=bs, num_workers=2),
                               output_dir=str(Path(cfg.paths.runs_dir)/run_id), device=DEVICE)
        m = evaluate(run_cfg, run_id, ckpt, DataLoader(te, batch_size=bs, num_workers=2),
                     results_dir=cfg.paths.results_dir, device=DEVICE)
        registry.set_status(run_id, 'done', result_path=str(Path(cfg.paths.runs_dir)/run_id))
        print(f'[{run_id}] HC-IoU={m.get("hc_iou", float("nan")):.4f}')
    except Exception as e:
        registry.set_status(run_id, 'failed')
        import traceback; traceback.print_exc()
print(f'\nDone. Registry: {registry.summary()}')

In [ ]:
# [GPU]  A100 if GRAM model needs inference; CPU if reading cached GRAM outputs
# [TIME] ~5–10 min
# [CU]   low; only if D6 enabled and GRAM outputs exist
# Cell P4.3: GRAM head-to-head on HC tiles (optional, Decision D6)
if base_cfg.get('decisions', {}).get('D6_include_gram_headtohead', True):
    GRAM_DIR = str(PROJECT_ROOT / 'gram_baseline')
    if Path(GRAM_DIR).exists():
        print('GRAM outputs found — assemble labels tensor from test set and call evaluate_gram().')
        print('TODO_VERIFY: wire GRAM output filenames to load_gram_predictions() format.')
    else:
        print(f'No GRAM outputs at {GRAM_DIR} — skipping head-to-head.')
else:
    print('GRAM head-to-head disabled (D6=false).')

## Phase 5 — Figures & Tables  *(switch back to CPU to save CU)*

In [ ]:
# [GPU]  CPU is fine
# [TIME] ~1 min
# [CU]   ~0
# Cell P5.1: Regenerate all figures + LaTeX tables + RESULTS.md (from results CSV)
subprocess.run([sys.executable, 'scripts/make_paper_figures.py',
                '--results_dir', cfg.paths.results_dir], check=True)
print('Figures →', str(Path(cfg.paths.results_dir)/'figures'))
print('Tables  →', str(Path(cfg.paths.results_dir)/'tables'))

In [ ]:
# [GPU]  CPU is fine
# [TIME] instant
# [CU]   ~0
# Cell P5.2: Display all figures inline
from IPython.display import Image as IPyImage, display
for fig_path in sorted((Path(cfg.paths.results_dir) / 'figures').glob('*.png')):
    print(f'\n--- {fig_path.name} ---')
    display(IPyImage(str(fig_path)))

In [ ]:
# [GPU]  CPU is fine
# [TIME] instant
# [CU]   ~0
# Cell P5.3: Headline results + confirm everything is on Drive for local pull
import pandas as pd
csv_path = str(Path(cfg.paths.results_dir) / 'tables' / 'all_runs.csv')
if Path(csv_path).exists():
    df = pd.read_csv(csv_path)
    cols = ['run_id', 'metric_hc_iou', 'metric_precision', 'metric_recall',
            'metric_f1', 'metric_fpr_control_old_dhaka']
    avail = [c for c in cols if c in df.columns]
    print(df[avail].sort_values('metric_hc_iou', ascending=False).to_string(index=False))
print('\n=== Saved to Drive (pull these into your local repo for the paper) ===')
print('  results/tables/all_runs.csv      <- every run, one row each')
print('  results/tables/master_table.tex  <- LaTeX, paste into IEEE template')
print('  results/runs/*.json              <- per-run provenance (config hash, seed, git)')
print('  results/figures/*.png / *.pdf    <- paper figures (PDF for LaTeX)')
print('  docs/RESULTS.md                  <- auto-updated markdown summary')